In [1]:
# In Colab: Runtime > Change runtime type > T4 GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
%%bash
# Install bitsandbytes from the pre-built CUDA wheel directly
pip install -q \
    "bitsandbytes==0.42.0" \
    "triton==2.2.0" \
    "transformers==4.41.0" \
    "peft==0.10.0" \
    "accelerate==0.29.3" \
    "datasets==2.19.0" \
    "trl==0.8.6"

In [3]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


Let's try installing a compatible `bitsandbytes` version. This will override the previous installation.

In [4]:
!python -m bitsandbytes

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
++++++++++++++++++ BUG REPORT INFORMATION ++++++++++++++++++
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

++++++++++++++++++ /usr/local CUDA PATHS +++++++++++++++++++
/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda118_nocublaslt.so
/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda122.so
/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda123.so
/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda120.so
/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda117_nocublaslt.so
/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda110_nocublaslt.so
/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda117.so
/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda111.so
/usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda118.so
/u

In [5]:
%%bash
# Install bitsandbytes from the CUDA 12.x wheel index
pip install -q --no-cache-dir bitsandbytes --prefer-binary --extra-index-url https://huggingface.github.io/bitsandbytes-wheels/cuda12x/

After running the above cell, you might need to restart your Colab runtime (`Runtime` > `Restart session`) for the new `bitsandbytes` installation to take effect, especially for low-level CUDA bindings.

In [6]:
from huggingface_hub import notebook_login
notebook_login()


# Model and Tokenizer Loading

In [11]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

def get_bnb_config() -> BitsAndBytesConfig:
  return BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_use_double_quant=True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_compute_dtype=torch.bfloat16,
  )

In [12]:
def load_model_and_tokenizer(model_name: str, device_map:str="auto"):
  bnb_config = get_bnb_config()
  model = AutoModelForCausalLM.from_pretrained(
      model_name,
      device_map=device_map,
      trust_remote_code=True,
      quantization_config=bnb_config,
  )

  model.config.use_cache = False
  model.gradient_checkpointing_enable()

  tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
  if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

  tokenizer.padding_side = "right"

  print(f"Model loaded. trainable params: {count_trainable_params(model)}")
  return model, tokenizer


def count_trainable_params(model) -> str:
  trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
  total = sum(p.numel() for p in model.parameters())
  return f"{trainable:,}/{total:,} ({100 * trainable / total:.2f}%)"


# LoRA configuration

In [13]:


from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

def get_lora_config(
    r: int = 16,
    lora_alpha: int = 32,
    lora_dropout: float = 0.05,
    target_modules: list = None,
) -> LoraConfig:

    if target_modules is None:

        target_modules = [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ]

    return LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        target_modules=target_modules,
        lora_dropout=lora_dropout,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )


def apply_lora(model, lora_config: LoraConfig):

    # Step 1: Prepare quantized model for k-bit training
    # This MUST be called before get_peft_model for 4-bit models
    model = prepare_model_for_kbit_training(model)

    # Step 2: Inject LoRA adapters
    model = get_peft_model(model, lora_config)

    # Confirm what's trainable
    model.print_trainable_parameters()
    # Output should look like:
    # trainable params: 5,636,096 || all params: 1,105,199,104 || trainable%: 0.5100

    return model

# Dataset and Tokenisation

In [15]:


from datasets import load_dataset
from transformers import PreTrainedTokenizer
from typing import Dict

ALPACA_PROMPT = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

# End-of-sequence token
EOS_TOKEN = None


def format_alpaca_sample(sample: Dict) -> Dict:
    """
    Format a single dataset sample into the instruction-tuning template.
    We add the EOS token so the model learns when to stop generating.
    """
    text = ALPACA_PROMPT.format(
        instruction=sample.get("instruction", ""),
        input=sample.get("input", ""),
        output=sample.get("output", "")
    ) + EOS_TOKEN
    return {"text": text}


def load_and_prepare_dataset(
    dataset_name: str = "yahma/alpaca-cleaned",
    tokenizer: PreTrainedTokenizer = None,
    max_length: int = 512,
    num_samples: int = 5000,   # Use subset for Colab speed
    val_split: float = 0.1,
):

    global EOS_TOKEN
    EOS_TOKEN = tokenizer.eos_token

    # Load dataset
    dataset = load_dataset(dataset_name, split="train")

    # Use a subset to fit Colab time constraints
    dataset = dataset.select(range(min(num_samples, len(dataset))))

    # Format into instruction template
    dataset = dataset.map(format_alpaca_sample)

    # Tokenize
    def tokenize(sample):

        result = tokenizer(
            sample["text"],
            truncation=True,
            max_length=max_length,
            padding="max_length",
        )
        result["labels"] = [
            token_id if token_id != tokenizer.pad_token_id else -100
            for token_id in result["input_ids"]
        ]
        return result

    dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

    # Train/validation split
    split = dataset.train_test_split(test_size=val_split, seed=42)

    print(f"Train: {len(split['train'])} | Val: {len(split['test'])}")
    print(f"Sample input length: {sum(1 for x in split['train'][0]['input_ids'] if x != tokenizer.pad_token_id)} tokens")

    return split['train'], split['test']

# Training Pipeline

In [20]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from trl import SFTTrainer
import os

def get_training_args(output_dir: str = "./results") -> TrainingArguments:
    return TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=1,
        per_device_eval_batch_size = 1,
        gradient_accumulation_steps = 4,
        learning_rate = 2e-4,
        num_train_epochs=3,
        lr_scheduler_type="cosine",
        warmup_ratio=0.3,

        fp16=True,
        gradient_checkpointing=True,
        optim="paged_adamw_32bit",
        weight_decay=0.001,
        max_grad_norm=0.3,

        # Logging & saving
        logging_steps=10,
        save_steps=50,
        evaluation_strategy="steps",
        eval_steps=50,
        save_total_limit=3,
        load_best_model_at_end=True,

        # Reporting
        report_to="none",
        seed=42,
    )


def train_model(model, tokenizer, train_dataset, eval_dataset, output_dir="./results"):
  training_args = get_training_args(output_dir)

  trainer = SFTTrainer(
      model=model,
      tokenizer=tokenizer,
      train_dataset=train_dataset,
      eval_dataset=eval_dataset,
      args=training_args,
      dataset_text_field="text",
      max_seq_length=512,
      packing=False,
  )

  before_mem = torch.cuda.memory_allocated() / 1e9
  print(f"GPU memory before training: {before_mem:.2f} GB")

  # ---- TRAIN ----
  trainer.train()

  after_mem = torch.cuda.memory_allocated() / 1e9
  print(f"GPU memory after training: {after_mem:.2f} GB")

  # Save the final adapter weights
  # This saves ONLY the LoRA adapter (~10-40MB), not the full model!
  trainer.save_model(output_dir)
  tokenizer.save_pretrained(output_dir)
  print(f"Adapter saved to {output_dir}")

  return trainer

# Entry Point

In [25]:


import torch

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./qlora-finetuned"

def main():
    print("=" * 60)
    print("QLoRA Fine-Tuning Pipeline")
    print("=" * 60)

    # 1. Load quantized model
    print("\n[1/4] Loading quantized model...")
    model, tokenizer = load_model_and_tokenizer(MODEL_NAME)

    # 2. Apply LoRA adapters
    print("\n[2/4] Applying LoRA adapters...")
    lora_config = get_lora_config(r=16, lora_alpha=32, lora_dropout=0.05)
    model = apply_lora(model, lora_config)

    # 3. Load and prepare dataset
    print("\n[3/4] Preparing dataset...")
    train_data, val_data = load_and_prepare_dataset(
        tokenizer=tokenizer,
        num_samples=1000,
        max_length=512,
    )

    # 4. Train
    print("\n[4/4] Starting training...")
    trainer = train_model(model, tokenizer, train_data, val_data, OUTPUT_DIR)

    print("\nTraining complete!")
    print(f"Model saved to: {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

QLoRA Fine-Tuning Pipeline

[1/4] Loading quantized model...
Model loaded. trainable params: 131,164,160/615,606,272 (21.31%)

[2/4] Applying LoRA adapters...
trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338264987769031

[3/4] Preparing dataset...


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train: 900 | Val: 100
Sample input length: 512 tokens

[4/4] Starting training...
GPU memory before training: 2.06 GB


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss,Validation Loss
50,1.018800,1.039987
100,0.912600,0.991710
150,0.975500,0.981319
200,0.985700,0.970495
250,0.814900,0.978629
300,0.905100,0.975673
350,0.861200,0.967554
400,0.880800,0.970570
450,0.850500,0.966287
500,0.489200,1.028215


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download

GPU memory after training: 1.56 GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Adapter saved to ./qlora-finetuned

Training complete!
Model saved to: ./qlora-finetuned


Evaluation

In [26]:


import torch
import math
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def load_finetuned_model(base_model_name: str, adapter_path: str):

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    base = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(adapter_path)

    # Load the LoRA adapter on top of the base model
    model = PeftModel.from_pretrained(base, adapter_path)
    model.eval()  # disable dropout

    return model, tokenizer


def compute_perplexity(model, tokenizer, texts: list, max_length: int = 512) -> float:

    total_loss = 0
    total_tokens = 0

    with torch.no_grad():
        for text in texts:
            enc = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=max_length
            ).to(model.device)

            outputs = model(**enc, labels=enc["input_ids"])
            loss = outputs.loss
            num_tokens = enc["input_ids"].shape[1]

            total_loss += loss.item() * num_tokens
            total_tokens += num_tokens

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    return perplexity


def generate_response(model, tokenizer, instruction: str, max_new_tokens: int = 200) -> str:

    prompt = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:


### Response:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,         # Controls randomness: 0=greedy, 1=sampling
            top_p=0.9,               # Nucleus sampling: consider top 90% probability mass
            do_sample=True,          # Use sampling (not greedy decoding)
            repetition_penalty=1.1,  # Penalize repeated phrases
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()


if __name__ == "__main__":
    model, tokenizer = load_finetuned_model(
        "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "./qlora-finetuned"
    )

    # Test instructions
    test_prompts = [
        "Explain the difference between supervised and unsupervised learning.",
        "Write a Python function to reverse a linked list.",
        "What are three advantages of renewable energy?",
    ]

    for prompt in test_prompts:
        print(f"\nInstruction: {prompt}")
        print(f"Response: {generate_response(model, tokenizer, prompt)}")
        print("-" * 60)

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565



Instruction: Explain the difference between supervised and unsupervised learning.
Response: Supervised learning refers to learning tasks where the output is known in advance and can be classified as either positive or negative examples. In contrast, unsupervised learning does not require any input or labels for training and focuses on discovering patterns within data without providing explicit labels. 

Supervised learning is often used in machine learning algorithms such as classification and regression, where we provide labeled examples to the algorithm and it learns from them to make predictions about new data. For example, if we have a dataset of customer behavior data, we might use supervised learning algorithms like decision trees or neural networks to predict the likelihood of purchasing something based on past behaviors.

On the other hand, unsupervised learning techniques are less common but still useful in many applications, including natural language processing, image recog

In [27]:


from peft import PeftModel
import torch

def save_adapter(model, tokenizer, save_path: str):
    """
    Save only the LoRA adapter weights (~10–50MB).
    The base model is NOT saved — only the small delta.
    """
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Adapter saved to: {save_path}")



def merge_and_save(model, tokenizer, save_path: str):

    # Merge adapter weights into base model
    merged_model = model.merge_and_unload()

    # Save in half precision to save disk
    merged_model.save_pretrained(save_path, safe_serialization=True)
    tokenizer.save_pretrained(save_path)
    print(f"Merged model saved to: {save_path}")


def push_to_hub(model, tokenizer, repo_name: str):

    model.push_to_hub(repo_name, use_auth_token=True)
    tokenizer.push_to_hub(repo_name, use_auth_token=True)
    print(f"Pushed to: https://huggingface.co/{repo_name}")